# Maximize Current through U-LEBT two Apertures

In [1]:
import time
import datetime
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import OrderedDict
from epics import caput, caget, caget_many
# from epics import caget, caget_many

In [2]:
import sys
repo_root = '/user/shared/pkgs/stBO'
sys.path.insert(0, str(repo_root))
from stbo.optimization import BOController
from stbo.utils import live_monitor_plot, live_history_plot

In [3]:
repo_root = '/user/shared/pkgs/machineIO'
sys.path.insert(0, str(repo_root))
from machineIO import construct_machineIO, Evaluator, OracleEvaluator
from machineIO.objFunc import SingleTaskObjectiveFunction
from machineIO import preset

# Prepare and Check machine status

##### check source and chopper setting

In [4]:
SCS = caget("ACS_DIAG:DEST:ACTIVE_ION_SOURCE")
ion = caget("FE_ISRC"+str(SCS)+":BEAM:ELMT_BOOK")
Q = caget("FE_ISRC"+str(SCS)+":BEAM:Q_BOOK")
A = caget("FE_ISRC"+str(SCS)+":BEAM:A_BOOK")
# AQ = caget("FE_ISRC2:BEAM:MOVRQ_BOOK")
AQ = A/Q
ion = str(A)+ion+str(Q)
print('SCS'+str(SCS), ion, 'A/Q=',AQ)

duty = np.round(caget('GTS_FTS:MSTR_N0001:PCUR_DFAC_RD'),decimals=4)
mode = caget('GTS_FTS:MSTR_N0001:MSG_RD_FSM')
rep = caget('GTS_FTS:MSTR_N0001:FR_CSET_REP')

print(f'Mode: {mode}')
print(f'Rep Rate: {rep}Hz, Duty: {duty}%')

SCS2 238U37 A/Q= 6.4324324324324325
Mode: O8.1 FE Commissioning 100Hz
Rep Rate: 100.0Hz, Duty: 25.0%


##### check upstream FC

In [5]:
upstream_FC_isoutPVs  = ['FE_SCS2:FC_D0717:LMPOS_RSTS_DRV']
upstream_FC_insertPVs = ['FE_SCS2:FC_D0717:IN_CMD_DRV']
isCMDsent = False
for FC_isoutPV, FC_insertPV in zip(upstream_FC_isoutPVs, upstream_FC_insertPVs):
    is_FC_out = caget(FC_isoutPV)
    if not is_FC_out:
        r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is inserted. Do you allow me to take it out? ")
        if r in ['yes','y','YES','Y']:
            caput(FC_insertPV,0)
            isCMDsent = True
if isCMDsent:
    time.sleep(10)
for FC_isoutPV, FC_insertPV in zip(upstream_FC_isoutPVs, upstream_FC_insertPVs):
    is_FC_out = caget(FC_isoutPV)
    if not is_FC_out:
        r = input(f"FC {FC_insertPV.strip(':IN_CMD')} is still inserted. please check manually. enter to continue")

##### check FC998

In [6]:
is_FC_out = caget('FE_LEBT:FC_D0998:LMPOS_RSTS_DRV')
if is_FC_out:
        r = input("FE_LEBT:FC_D0998 is out. Do you allow me to insert it? ")
        if r in ['yes','y','YES','Y']:
            caput('FE_LEBT:FC_D0998:IN_CMD',1)
            time.sleep(5)
            is_FC_out = caget('FE_LEBT:FC_D0998:LMOUT_RSTS')
            if is_FC_out:
                r = input(f"FE_LEBT:FC_D0998 is still not inserted. please check manually. enter to continue")

# check FC814 range 1055 uA
if caget('FE_LEBT:FC_D0998:RNG_CMD') != 0:
    r = input("FE_LEBT:FC_D0998 range is set to 1uA. Do you want me to change too 1055 uA? ")
    if r in ['yes','y','YES','Y']:
        caput('FE_LEBT:FC_D0998:RNG_CMD',0)
        time.sleep(5)
        if caget('FE_LEBT:FC_D0998:RNG_CMD') != 0:
            r = input(f"FE_LEBT:FC_D0998 range is still not correct. please check manually.")

cannot connect to FE_LEBT:FC_D0998:LMPOS_RSTS_DRV


##### check apertures

In [7]:
aperture_setPVs = ['FE_LEBT:AP_D0796:IN_CMD','FE_LEBT:AP_D0807:IN_CMD']
aperture_rdPVs  = ['FE_LEBT:AP_D0796:LMIN_RSTS','FE_LEBT:AP_D0807:LMIN_RSTS']
aperture_rd_targets = [1, 1]
aperture_rd_tols    = [0.1, 0.1]
isCMDsent = False
for i,pv in enumerate(aperture_rdPVs):
    pv_simple = pv.strip(':LMIN_RSTS').strip(':IN_CMD')
    target = aperture_rd_targets[i]
    val = caget(pv)
    tol = aperture_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"aperture {pv_simple} is not inserted. Do you allow me to put it in? ")
        if r in ['yes','y','YES','Y']:
            caput(aperture_setPVs[i],target)
            print(f"aperture {pv_simple} is inserted")
if isCMDsent:
    time.sleep(3)
for i,pv in enumerate(aperture_rdPVs):
    pv_simple = pv.strip(':LMIN_RSTS').strip(':IN_CMD')
    target = aperture_rd_targets[i]
    val = caget(pv)
    tol = aperture_rd_tols[i]
    isin = target-tol < val < target+tol
    if not isin:
        r = input(f"aperture {pv_simple} is still not inserted. please check manually. enter to continue")

##### Turn off correctors b/w Apertures

In [8]:
corrector_setPVs = ["FE_LEBT:PSC2_D0805:I_CSET","FE_LEBT:PSC1_D0805:I_CSET"]
corrector_rdPVs  = [pv.replace('_CSET','_RD') for pv in corrector_setPVs]
corrector_rd_targets = np.array([0]*len(corrector_setPVs))
corrector_rd_tols    = np.array([0.05]*len(corrector_setPVs))

rd_vals = caget_many(corrector_rdPVs)
if np.any(np.abs(rd_vals - corrector_rd_targets)/corrector_rd_tols > 1):
    display(pd.DataFrame(rd_vals,index=corrector_rdPVs,columns=['val']).T)
    r = input(f"correctors b/w Apertures are not off. Do you allow me to set it to zero? ")
    if r in ['yes','y','YES','Y']:
        time.sleep(2)
        rd_vals = caget_many(corrector_rdPVs)
        caput_many(corrector_setPVs,corrector_rd_targets)
        if np.abs(rd_vals - corrector_rd_targets)/corrector_rd_tols > 1:
            print("correctors b/w Apertures set to zero. But read-back shows sitll not off. please check manually")
        else:
            print("correctors b/w Apertures set to zero.")

# === Begin User Inputs ===
##### Define current at 100% transmission (source FC reading)

In [9]:
maxBeamCurrent = 45.4

##### IO settings

In [10]:
data_acquire_rate = 5 # [Hz]
timespan_for_average = 2.0  # [sec]
additional_wait_after_powersupply_ramp  = 0.3 # [sec]

##### Optimizer setting 
Do not exceed budget 100. It will impractically slow down training and quering GP model

In [11]:
is_close_to_opt = True

if is_close_to_opt:
    n_init_budget       = 4   # recommended: order of number of decision parameters
    n_global_opt_budget = 0
    n_local_opt_budget  = 30
    n_finetune_budget   = 4   # recommended: order of number of decision parameters 
else:
    n_init_budget       = 20          
    n_global_opt_budget = 20
    n_local_opt_budget  = 20
    n_finetune_budget   = 3    # recommended: less than number of decision parameters 

_budget = n_init_budget +n_global_opt_budget +n_local_opt_budget +n_finetune_budget
_ramping_time = 1 # on average, each iter, considering polarity crossing time
_expected_oracle_time_cost = timespan_for_average + additional_wait_after_powersupply_ramp + _ramping_time
print(f"budget: {_budget}")
print(f"expected run time: {int(_budget*_expected_oracle_time_cost)} sec")

budget: 38
expected run time: 125 sec


### Define controls
to do list:
 - try a V-dipole instead of a V-corrector to avoid polarity crossing time
 - try solenoids 

In [12]:
control_CSETs= [
    # 'FE_SCS2:PSC2_D0731:I_CSET', 'FE_SCS2:PSC1_D0731:I_CSET',
    'FE_SCS2:PSC2_D0755:I_CSET', 'FE_SCS2:PSC1_D0755:I_CSET',
    # 'FE_LEBT:PSC2_D0773:I_CSET', 'FE_LEBT:PSC1_D0773:I_CSET',
    'FE_LEBT:PSC2_D0790:I_CSET', 'FE_LEBT:PSC1_D0790:I_CSET',
    # 'FE_LEBT:PSOL_D0787:I_CSET',
]
control_RDs  = [pv.replace('_CSET','_RD') for pv in control_CSETs]

is_cryo = np.any(['_CB' in PV or '_CA' in PV or  '_CC' in PV or  '_CD' in PV or  '_CE' in PV for PV in control_CSETs])
if not is_close_to_opt and is_cryo:
    print('warn: global optimization is proposed with controls in croyo module. proceed with care')

In [13]:
x0 = caget_many(control_CSETs)

control_tols = []
control_min = []
control_max = []
for v, PV in zip(x0,control_CSETs):
    if 'PSC' in PV:
        control_min.append(v-0.8*AQ)
        control_max.append(v+0.8*AQ)
#         control_min.append( -0.8*AQ)
#         control_max.append( +0.8*AQ)
        control_tols.append(0.2)
    elif 'PSOL' in PV:
        control_min.append(0.95*v)
        control_max.append(1.05*v)
        control_tols.append(1.0)
    else:
        raise ValueError(f'control bounds for {PV} cannot be determined')

assert len(control_CSETs) == len(control_min) == len(control_max) == len(control_tols)
control_Lo_limit, control_Hi_limit = preset.get_limits(control_CSETs)
control_min = np.clip(control_min, a_min = control_Lo_limit, a_max = None)
control_max = np.clip(control_max, a_min = None, a_max = control_Hi_limit)
assert np.all(control_max > control_min)
control_bounds = torch.tensor([control_min.tolist(), control_max.tolist()], dtype=torch.float64)

print("============== check control bounds ================= ")
pd.DataFrame(np.array([x0,control_min,control_max,control_tols,control_Lo_limit,control_Hi_limit]).T,
             index=control_CSETs, 
             columns=['current value','control min','control max','tol','LoLim','HiLim'])

============== check control bounds ================= 


,current value,control min,control max,tol,LoLim,HiLim
FE_SCS2:PSC2_D0755:I_CSET,1.000,-4.145946,5.5,0.2,-5.5,5.5
FE_SCS2:PSC1_D0755:I_CSET,0.813,-4.332946,5.5,0.2,-5.5,5.5
FE_LEBT:PSC2_D0790:I_CSET,2.502,-2.643946,5.5,0.2,-5.5,5.5
FE_LEBT:PSC1_D0790:I_CSET,1.368,-3.777946,5.5,0.2,-5.5,5.5


### Define objectives

In [14]:
objective_goal   = {'FE_LEBT:FC_D0998:PKAVG_RD': {'more than': maxBeamCurrent}}
objective_weight = {'FE_LEBT:FC_D0998:PKAVG_RD': 1}
objective_tolerance   = {'FE_LEBT:FC_D0998:PKAVG_RD': 0.2*maxBeamCurrent}
objective_weight = {k:v for k,v in objective_weight.items() if v!=0}
objective_PVs = list(objective_weight.keys())

print("============== check objective ================= ")
print(" too small numbers may rounded to show 0 but actual value may not be 0")
pd.DataFrame([objective_goal,objective_tolerance,objective_weight],index=['goal','norm','weight']).T

============== check objective ================= 
 too small numbers may rounded to show 0 but actual value may not be 0


,goal,norm,weight
FE_LEBT:FC_D0998:PKAVG_RD,{'more than': 45.4},9.08,1


###### preprare live plot  --> specify PV groups (list of list) to plot togather
monitors

In [15]:
extra_monitors = []
# monitors
monitor_PVs = objective_PVs + extra_monitors
CURRENTs = [pv for pv in monitor_PVs if '_RD' in pv and (':FC_D' in pv or ':BCM_D' in pv) ]
POSs = [pv for pv in monitor_PVs if ':BPM_D' in pv and 'POS_RD' in pv]
PHASEs = [pv for pv in monitor_PVs if 'PHASE' in pv]
monitor_groups = [CURRENTs,POSs,PHASEs]

controls

In [16]:
CORs = [pv for pv in control_CSETs if ':PSC' in pv]
SOLs = [pv for pv in control_CSETs if ':PSOL' in pv]
control_CSETs_groups = [CORs,SOLs]
monitor_groups = [l for l in monitor_groups if len(l)>0]

CORs = [pv for pv in control_RDs if ':PSC' in pv]
SOLs = [pv for pv in control_RDs if ':PSOL' in pv]
control_RDs_groups = [CORs,SOLs]

# === End of User Inputs ===

In [17]:
now0 = datetime.datetime.now()
fname = now0.strftime('%Y%m%d_%H%M')+'['+ion+'][stBO][SCS2-ULEBT]FC0998'
fname

'20260706_1036[238U37][stBO][SCS2-ULEBT]FC0998'

##### Prepare machine evaluator

In [18]:
_expected_bo_computation_time = 1 # [sec]
_run_async = 0.5 < abs(_expected_oracle_time_cost/_expected_bo_computation_time - 1) < 2
print(f"_run_async: {_run_async}")

machineIO = construct_machineIO(
    fetch_data_time_span = timespan_for_average,
    ensure_set_timewait_after_ramp = additional_wait_after_powersupply_ramp,
    sample_interval = 1/data_acquire_rate,
    test = False,
)

_run_async: False


In [19]:
composite_objective_name = 'composite_obj'
obj_func = SingleTaskObjectiveFunction(
    objective_PVs = objective_PVs,
    composite_objective_name = composite_objective_name,
    objective_goal = objective_goal,
    objective_weight = objective_weight,
    objective_tolerance = objective_tolerance,
    p_order = 1,
    apply_bilog = False
)

In [20]:
oracle_key_names = {
    'x':control_RDs,
    'y':composite_objective_name,
}

oracle =  OracleEvaluator(
    machineIO,
    control_CSETs= control_CSETs,
    control_RDs  = control_RDs,
    control_tols = control_tols,
    oracle_key_names = oracle_key_names,
    monitor_PVs  = monitor_PVs,
    df_manipulators = [obj_func.calculate_objectives_from_df],
)

##### preprare live plot

In [21]:
bo = BOController(oracle, bounds = control_bounds)

[10:36:36.016] WARNING: phantasy.~.epics_tools: Established 9 PVs in 29.3 ms.


In [22]:
monitor_live = live_monitor_plot(
    bo, oracle,
    monitor_groups       = [l for l in monitor_groups if len(l)>0], 
    control_CSETs_groups = [l for l in control_CSETs_groups if len(l)>0],
    control_RDs_groups   = [l for l in control_RDs_groups if len(l)>0],
)
history_live = live_history_plot(bo)
monitor_live.start()
history_live.start()

/user/shared/pkgs/stBO/stbo/utils/live_plots.py:270: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim(y_min - margin, y_max + margin)


# run

In [23]:
print("initializatin...")
bo.initialize(budget=n_init_budget, local_init=is_close_to_opt)

print("global optimization...")
for _ in range(n_global_opt_budget):
    fresh_train = True #(not bool(_%2)) or len(bo.train_x) < bo.bounds.shape[1]*2
    bo.step(mode="global", acq_type="qEI", fresh_train=fresh_train, plot_acq=False, asynchro=_run_async)
    
print("local optimization...")
for _ in range(n_local_opt_budget):
    fresh_train = True #(not bool(_%2)) or len(bo.train_x) < bo.bounds.shape[1]*2
    bo.step(mode="local", acq_type="qEI", fresh_train=fresh_train, plot_acq=False, asynchro=_run_async)
    
print("fine tuning...")
for _ in range(n_finetune_budget):
    bo.step(mode="fine_tune", acq_type="qEI", fresh_train=True, plot_acq=False, asynchro=_run_async)
    plt.show()

initializatin...
[10:36:39.933] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:36:44.115] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:36:48.005] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
global optimization...
local optimization...
[10:36:52.609] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:36:57.314] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:01.303] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:05.310] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:11.940] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:18.011] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:24.005] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:28.698] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:34.207] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
[10:37:42.228]

In [24]:
now1 = datetime.datetime.now()
print(f"optimization took {(now1-now0).seconds} sec")

optimization took 257 sec


# Set to Best solution 

In [25]:
x_hist = [h["x"] for h in bo.history]
y_hist = [h["y"] for h in bo.history]
imax = np.argmax(y_hist)

orcle_dic = oracle(x_hist[imax])

print("Best x:", x_hist[imax])
print("Best y (old):", y_hist[imax])
print("Best y (new):", orcle_dic['y'])

[10:41:00.317] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
Best x: [-0.72763   0.018134  0.75686   0.84296 ]
Best y (old): [-0.60643089]
Best y (new): [-0.61895317]


### comparision: before and after opt

In [26]:
tmp = np.vstack((x_hist[0],x_hist[imax]))
pd.DataFrame(tmp,columns=control_CSETs,index=['before opt','after opt']).T

,before opt,after opt
FE_SCS2:PSC2_D0755:I_CSET,1.416910,-0.727630
FE_SCS2:PSC1_D0755:I_CSET,1.196070,0.018134
FE_LEBT:PSC2_D0790:I_CSET,2.600539,0.756860
FE_LEBT:PSC1_D0790:I_CSET,1.079060,0.842960


[10:41:20.317] WARNING: phantasy.~.epics_tools: Established 9 PVs in 0.1 ms.
